# 10b -- Monocle3 轨迹推断 (独立 Notebook)

本 notebook 对上皮谱系做 **Monocle3 拟时序分析**，是拟时序方法系列的第一个独立 notebook。
与 `10_pseudotime.ipynb`（综合 notebook：熵 + CytoTRACE + root 识别 + Monocle3）并行存在，
PI 可分别跑、分别看、最后横向比较不同方法的 pseudotime 结果。

## 方法原理

**Monocle3** 是基于图学习的轨迹推断工具：
1. 从细胞表达数据构建 principal graph（UMAP 降维 + 聚类 + `learn_graph`）
2. 在图上识别叶节点（leaf nodes）和分支点（branch points）
3. 从用户指定的 root cells 出发，沿图计算每个细胞的 pseudotime
4. 输出：pseudotime 排序 + 轨迹图 + 聚类/分区信息

Monocle3 擅长处理复杂的分叉轨迹，是目前最广泛使用的拟时序工具之一。
本 notebook 通过 subprocess Rscript 桥接调用 R 版 Monocle3；R 不可用或 monocle3 包未安装时优雅跳过。

## 输入与输出

| 项目 | 路径/字段 |
|------|-----------|
| 上游输入 | `UPSTREAM_PATH`（默认 `results/06_annotated_v1.h5ad`）|
| Root 来源 | 优先读取上游 `adata.uns["root_cluster"]`（如 10_pseudotime 已跑）；否则用干细胞 marker 均值最高 cluster 作 fallback |
| 输出 h5ad | `OUTPUT_PATH`（默认 `results/10b_pseudotime_monocle3_v1.h5ad`）|
| 新增 obs 列 | `pseudotime_monocle3_v1`、`monocle3_cluster`、`monocle3_partition`、`monocle3_is_leaf`、`root_cluster` |
| 新增 obsm key | `X_monocle3_umap` |
| Figure | `results/figures/10b_monocle3_trajectory_pseudotime.png`、`..._partition.png` |

## 与其他拟时序 notebook 的关系

| Notebook | 方法 | 产物 obs 列 |
|----------|------|-------------|
| `10_pseudotime.ipynb` | 综合（熵 + CytoTRACE + Monocle3）| `entropy` / `cytotrace_score` / `pseudotime_monocle3_v1` |
| `10b_pseudotime_monocle3.ipynb`（本 notebook）| Monocle3 独立 | `pseudotime_monocle3_v1` |
| `10c_pseudotime_cellrank.ipynb`（计划中）| CellRank | `pseudotime_cellrank_v1` |

PI 可以在 Jupyter 中打开各 notebook 产出的 h5ad，交叉比较不同方法的 pseudotime 排序是否一致。


## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（细胞注释），读 `06_annotated_v*.h5ad`
- **同级**：`10_pseudotime.ipynb`（综合拟时序，含熵 + CytoTRACE + root 识别）
- **下游**：本 stage 产出 checkpoint 供 16_trajectory_de 或后续可视化使用

### 为什么要迭代回跑？
Monocle3 轨迹推断的质量取决于几个关键参数：
- **root cluster 选择**：决定 pseudotime 的起点和生物学解释力。如果上游 `10_pseudotime` 的 root 识别
  不合理，可在本 notebook 关闭 `USE_UPSTREAM_ROOT` 改用 stem marker fallback
- **PCA 降维维度**（`MONOCLE3_NUM_DIM`）：影响 principal graph 的拓扑结构
- **上皮筛选范围**（`EPITHELIAL_CLUSTERS` 或 `AUTO_SUBSET_EPITHELIAL`）

### 如何回跑（三步操作）
1. 改 `UPSTREAM_PATH`——指向要复用的上游文件版本
2. 改 `OUTPUT_PATH`——bump 版本号 `_v1` → `_v2`
3. 调整参数（在下方 `# === PARAMS ===` 区域）→ 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。旧版 `.h5ad` 文件不覆盖不删除
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）
- **`promoted`**：PI 审查后确认可传给下游使用的正式版本
- **下游取数**：后续 stage 的 `UPSTREAM_PATH` 指向你决定采用的版本即可

### 追溯链（自动写入 h5ad 的 `adata.uns`）
- `stage` = `"10b_pseudotime_monocle3"`
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号
- `10b_pseudotime_monocle3_v1` = 方法参数嵌套 dict


In [ ]:
# === PARAMS ===
# UPSTREAM_PATH           -- 06 注释结果 h5ad
# OUTPUT_PATH             -- 本 notebook 产出 checkpoint
# RUN_MONOCLE3            -- 布尔开关：PI 可关闭整段 Monocle3
# USE_UPSTREAM_ROOT       -- 优先使用上游 adata.uns["root_cluster"]（如 10_pseudotime 已跑）
# CLUSTER_KEY             -- 用于按簇聚合的 obs 列（root 识别 fallback 时使用）
# CELL_TYPE_COL           -- 优先使用的细胞类型列
# EPITHELIAL_CLUSTERS     -- 上皮谱系 cluster ID 列表；None=全体细胞
# AUTO_SUBSET_EPITHELIAL  -- 自动检测并筛选上皮细胞做轨迹分析
# EPITHELIAL_LABEL_COL    -- 从哪个列检测上皮标签
# EPITHELIAL_KEYWORDS     -- 上皮关键词列表（小写匹配）
# STEM_MARKERS            -- 已知干细胞/祖细胞 marker 基因列表（root fallback 时使用）
# MONOCLE3_WORK_DIR       -- Monocle3 Rscript 临时工作目录（独立，不与 10_pseudotime 冲突）
# MONOCLE3_NUM_DIM        -- Monocle3 PCA 降维维度
# MONOCLE3_CORES          -- Monocle3 并行线程数
# RSCRIPT_BIN             -- Rscript 可执行文件路径（通过 platform 模块统一解析）

UPSTREAM_PATH = "results/06_annotated_v1.h5ad"
OUTPUT_PATH   = "results/10b_pseudotime_monocle3_v1.h5ad"

# === Monocle3 开关 ===
RUN_MONOCLE3 = True  # PI 可设为 False 跳过整段 Monocle3

# === Root 识别 ===
USE_UPSTREAM_ROOT = True  # 优先使用上游 adata.uns["root_cluster"]；False=强制 stem marker fallback

CLUSTER_KEY   = "leiden_res_0.6"
CELL_TYPE_COL = "cell_type_final_v1"

# === 上皮筛选 ===
EPITHELIAL_CLUSTERS = None  # None=全体细胞；PI 按需设如 ["0", "3", "5"]

AUTO_SUBSET_EPITHELIAL = True
EPITHELIAL_LABEL_COL = "cell_type_final_v1"
EPITHELIAL_KEYWORDS = ["epithelial", "parietal", "chief", "mucous", "foveolar",
                       "pit", "neck", "spem", "im_", "intestinal", "goblet"]

# === Root fallback：干细胞 marker（仅在上游无 root_cluster 时使用）===
STEM_MARKERS = [
    "LGR5", "SOX9", "MKI67", "OLFM4", "TERT",
    "AXIN2", "LRIG1", "TFF2", "MUC6", "MUC5AC",
]
# 以上为胃上皮干细胞/祖细胞常用 marker。基因不存在时自动跳过。

# === Monocle3 参数 ===
MONOCLE3_WORK_DIR = "results/_monocle3_10b_tmp"  # 独立目录，不与 10_pseudotime 的 _monocle3_tmp 冲突
MONOCLE3_NUM_DIM  = 50
MONOCLE3_CORES    = 1

# RSCRIPT_BIN 通过 platform 模块统一解析
from scrna_integration.platform import check_r_available
RSCRIPT_BIN, _R_AVAILABLE = check_r_available()


In [ ]:
# === setup：sys.path + env_check + 导入 + 切换目录 ===
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/07_downstream/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc

_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")

# 环境自检（与其他 notebook 一致的 env_check 模式）
try:
    from scrna_integration.platform import env_check
    env_check(expected_env="scrna-integration")
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")

# 导入依赖
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import shutil
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}")


In [ ]:
# === 加载上游 h5ad ===
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")

# 确定实际使用的分组列
if CELL_TYPE_COL in adata.obs.columns:
    _group_col = CELL_TYPE_COL
    print(f"使用细胞类型列: {CELL_TYPE_COL}")
    _ct = adata.obs[CELL_TYPE_COL].dropna().astype(str)
    print(f"  细胞类型: {sorted(_ct.unique())}")
elif CLUSTER_KEY in adata.obs.columns:
    _group_col = CLUSTER_KEY
    print(f"CELL_TYPE_COL 不存在，fallback 到 CLUSTER_KEY: {CLUSTER_KEY}")
    print(f"  簇数: {adata.obs[CLUSTER_KEY].nunique()}")
else:
    raise KeyError(
        f"CELL_TYPE_COL '{CELL_TYPE_COL}' 和 "
        f"CLUSTER_KEY '{CLUSTER_KEY}' 都不在 obs 列中"
    )

# 检查 embedding
_has_umap = "X_umap" in adata.obsm
print(f"X_umap: {_has_umap}  |  obsm keys: {list(adata.obsm.keys())}")

# 上皮谱系筛选 -- EPITHELIAL_CLUSTERS 手动指定
if EPITHELIAL_CLUSTERS is not None:
    _epi_mask = adata.obs[_group_col].astype(str).isin(
        [str(c) for c in EPITHELIAL_CLUSTERS]
    )
    _n_before = adata.n_obs
    if not _epi_mask.any():
        raise ValueError(
            f"EPITHELIAL_CLUSTERS={EPITHELIAL_CLUSTERS} 不匹配任何细胞。"
            f"可用的 {_group_col} 值: "
            f"{sorted(adata.obs[_group_col].dropna().astype(str).unique())}"
        )
    adata = adata[_epi_mask].copy()
    print(
        f"上皮谱系筛选: {_n_before:,} -> {adata.n_obs:,} 细胞 "
        f"(保留 {_group_col}: {EPITHELIAL_CLUSTERS})"
    )
else:
    print(
        "EPITHELIAL_CLUSTERS=None，对所有细胞做拟时序。"
        "PI 可在完成细胞类型注释后指定上皮 cluster 重新分析。"
    )

# === 上皮自动筛选（P2-6）===
# 在手动筛选之外，提供自动关键词检测筛选路径。EPITHELIAL_CLUSTERS 已指定时跳过。
if AUTO_SUBSET_EPITHELIAL and EPITHELIAL_CLUSTERS is None:
    _original_n = adata.n_obs
    if EPITHELIAL_LABEL_COL in adata.obs.columns:
        _labels = adata.obs[EPITHELIAL_LABEL_COL].astype(str).str.lower()
        _epi_mask = _labels.apply(lambda x: any(kw in x for kw in EPITHELIAL_KEYWORDS))

        if _epi_mask.sum() > 50:
            print(f"上皮自动筛选: {_epi_mask.sum():,}/{_original_n:,} cells 匹配上皮关键词")
            adata = adata[_epi_mask].copy()
            print(f"  筛选后: {adata.n_obs:,} cells")
            _top_labels = adata.obs[EPITHELIAL_LABEL_COL].value_counts().head(10)
            print(f"  包含标签: {_top_labels.to_dict()}")
        else:
            print(f"WARNING: 仅 {_epi_mask.sum()} cells 匹配上皮关键词（< 50），跳过筛选")
            print(f"  → 检查 EPITHELIAL_KEYWORDS 或 EPITHELIAL_LABEL_COL 是否正确")
            AUTO_SUBSET_EPITHELIAL = False
    else:
        print(f"WARNING: {EPITHELIAL_LABEL_COL} 列不存在，跳过上皮筛选")
        AUTO_SUBSET_EPITHELIAL = False
elif not AUTO_SUBSET_EPITHELIAL and EPITHELIAL_CLUSTERS is None:
    print("AUTO_SUBSET_EPITHELIAL=False 且 EPITHELIAL_CLUSTERS=None，使用全部细胞")
elif EPITHELIAL_CLUSTERS is not None:
    print(f"EPITHELIAL_CLUSTERS 已手动指定 ({EPITHELIAL_CLUSTERS})，跳过自动筛选")


In [ ]:
# === Root Cluster 识别 ===
# Monocle3 的 order_cells() 需要指定 root cluster 作为 pseudotime 起点。
# 策略（两级）：
#   1. 优先：从上游 h5ad 读取 adata.uns["root_cluster"]（10_pseudotime 已跑则存在）
#   2. Fallback：用干细胞 marker 均值最高的 cluster 作为 root

_top_cluster = None  # 将传给 R 脚本的 root cluster
_present_markers = []  # 预初始化，确保所有分支都有定义

_root_from_upstream = False  # 追踪 root 来源，供输出 cell 使用
if USE_UPSTREAM_ROOT and "root_cluster" in adata.uns:
    _top_cluster = str(adata.uns["root_cluster"])
    _root_from_upstream = True
    print(f"从上游读取 root_cluster: {_top_cluster}")
    # 检查上游 root_cluster 是否在筛选后的 cluster 列表中
    _available_clusters = set(adata.obs[_group_col].astype(str).unique())
    if _top_cluster not in _available_clusters:
        print(f"WARNING: 上游 root_cluster '{_top_cluster}' 不在筛选后的 cluster 列表中")
        print(f"  可用 cluster: {sorted(_available_clusters)}")
        _top_cluster = None  # 触发 fallback
        _root_from_upstream = False  # 同步重置 provenance，确保 root_source 准确
else:
    if USE_UPSTREAM_ROOT:
        print("上游无 root_cluster（adata.uns['root_cluster'] 不存在），启用 stem marker fallback")
    else:
        print("USE_UPSTREAM_ROOT=False，强制使用 stem marker fallback")

# Fallback：干细胞 marker 均值最高的 cluster
if _top_cluster is None:
    _present_markers = [g for g in STEM_MARKERS if g in adata.var_names]
    _missing_markers = [g for g in STEM_MARKERS if g not in adata.var_names]
    print(f"干细胞 marker: {len(_present_markers)}/{len(STEM_MARKERS)} 个基因存在于数据集中")
    if _missing_markers:
        print(f"  缺失基因: {_missing_markers}")

    if len(_present_markers) == 0:
        raise ValueError(
            f"STEM_MARKERS 中所有基因都在数据集中不存在，无法做 root cluster fallback。"
            f"请：(1) 先跑 10_pseudotime.ipynb 产生 root_cluster；"
            f"或 (2) 调整 STEM_MARKERS 列表"
        )

    # 计算每细胞的干细胞 marker 均值（优先 counts layer）
    if "counts" in adata.layers:
        _stem_expr = np.asarray(
            adata[:, _present_markers].layers["counts"].mean(axis=1)
        ).ravel()
    elif adata.raw is not None:
        _stem_expr = np.asarray(
            adata.raw[:, _present_markers].X.mean(axis=1)
        ).ravel()
    else:
        _stem_expr = np.asarray(
            adata[:, _present_markers].X.mean(axis=1)
        ).ravel()

    # 按簇聚合干细胞 marker 均值，选最高的簇为 root
    # NaN 检测：astype(str) 会把 NaN 无声转为字面 "nan"
    if adata.obs[_group_col].isna().any():
        print("WARNING: _group_col 含 NaN 值，将转为字面 'nan' 可能误导 root 识别")
    _groups = adata.obs[_group_col].astype(str)
    _stem_by_cluster = pd.Series(_stem_expr, index=adata.obs_names).groupby(_groups).mean()
    _stem_by_cluster = _stem_by_cluster.sort_values(ascending=False)
    _top_cluster = _stem_by_cluster.index[0]
    print(f"\n干细胞 marker 均值最高 cluster = {_top_cluster}")
    print(f"各簇干细胞 marker 均值（前 5）:")
    for _clu, _val in _stem_by_cluster.head(5).items():
        print(f"  cluster {_clu}: {_val:.6f}")

    if _top_cluster == "nan":
        print("WARNING: root cluster 退化为 'nan'，上游注释列可能全为空值，请检查数据质量")

# 写入 adata（供 Monocle3 R 脚本通过 cell_meta CSV 读取）
adata.uns["root_cluster"] = str(_top_cluster)
adata.obs["root_cluster"] = adata.obs[_group_col].astype(str)
print(f"\nroot_cluster 已写入: adata.uns['root_cluster'] = '{_top_cluster}'")
print(f"  adata.obs['root_cluster'] 覆盖 {adata.n_obs:,} 个细胞")

## Monocle3 轨迹推断 (R subprocess)

**桥接方式**：Python 端导出 counts.mtx + cell_meta.csv + gene_anno.csv + existing_umap.csv，
内联生成 `run_monocle3.R` 脚本，`subprocess.run` 调用 Rscript 执行，
读回 `monocle3_cells.csv` 结果写入 adata.obs/obsm。

**优雅降级**：两层守卫——
1. `RUN_MONOCLE3` 布尔开关：PI 可设 `False` 跳过整段
2. R 环境守卫：Rscript 不可用或 monocle3 R 包未安装时自动跳过，不崩 notebook


In [ ]:
# Monocle3 R 环境守卫：检查 Rscript 和 monocle3 R 包是否可用。
# 模式：R 包不可用时优雅跳过，不崩 notebook。
# RSCRIPT_BIN 来自 PARAMS cell，已通过 check_r_available() 赋值；
# None 表示 Rscript 不可用，非 None 表示可用路径。

if RUN_MONOCLE3:
    _monocle3_available = False
    _r_available = RSCRIPT_BIN is not None
    print(f"Rscript 可用: {_r_available}  (RSCRIPT_BIN={RSCRIPT_BIN})")

    if _r_available:
        _check_cmd = [
            RSCRIPT_BIN, "--vanilla", "-e",
            "suppressPackageStartupMessages(library(monocle3)); cat('OK')",
        ]
        try:
            _res = subprocess.run(
                _check_cmd, capture_output=True, text=True, timeout=60,
            )
            if _res.returncode == 0 and "OK" in _res.stdout:
                _monocle3_available = True
                print("monocle3 R 包可用 -- Monocle3 轨迹推断将正常执行")
            else:
                print("monocle3 R 包不可用（load 失败或未安装）")
                print(f"  R stderr: {_res.stderr.strip()[:200]}")
        except (FileNotFoundError, subprocess.TimeoutExpired) as _e:
            print(f"Rscript 调用失败: {_e}")
    else:
        print("Rscript 不可用，Monocle3 将跳过")

    if not _monocle3_available:
        _monocle3_skip_msg = (
            "=" * 60 + "\n"
            "Monocle3 环境不可用，以下 cell 将优雅跳过。\n\n"
            "要启用 Monocle3 轨迹推断，请:\n"
            "  1. 确认 R 已安装（conda install -c conda-forge r-base）\n"
            "  2. 安装 monocle3 R 包:\n"
            "     R -e 'BiocManager::install(\"monocle3\")'\n"
            "  3. 重跑本 notebook\n"
            "=" * 60
        )
        print(_monocle3_skip_msg)
else:
    _monocle3_available = False
    print("RUN_MONOCLE3=False，Monocle3 整段跳过")


In [ ]:
# 导出 Monocle3 输入数据（.mtz + .csv）。
# Monocle3 需要原始计数矩阵（genes x cells 的 .mtz 格式）。
# 也导出 UMAP 坐标，让 Monocle3 直接复用 Python 侧的 embedding。

if RUN_MONOCLE3 and _monocle3_available:
    shutil.rmtree(MONOCLE3_WORK_DIR, ignore_errors=True)
    os.makedirs(MONOCLE3_WORK_DIR, exist_ok=True)

    # 获取计数矩阵（优先 counts layer）
    if "counts" in adata.layers:
        _X_export = adata.layers["counts"]
    elif adata.raw is not None:
        _X_export = adata.raw[:, adata.var_names].X
    else:
        _X_export = adata.X

    # 写 .mtz（genes x cells -- Monocle3 需要基因在行）
    from scipy.io import mmwrite
    _X_gc = sp.csr_matrix(_X_export).T.tocoo()
    _mtx_path = os.path.join(MONOCLE3_WORK_DIR, "counts.mtx")
    mmwrite(_mtx_path, _X_gc)
    print(f"计数矩阵已导出: {_mtx_path}  (shape genes x cells: {_X_gc.shape})")

    # 细胞元数据
    _meta = adata.obs.copy()
    _meta["cell_id"] = adata.obs_names.astype(str)
    for _col in _meta.columns:
        if hasattr(_meta[_col], "cat"):
            _meta[_col] = _meta[_col].astype(str)
    _meta_path = os.path.join(MONOCLE3_WORK_DIR, "cell_meta.csv")
    _meta.to_csv(_meta_path, index=False)
    print(f"细胞元数据已导出: {_meta_path}  ({_meta.shape[0]} 细胞 x {_meta.shape[1]} 列)")

    # 基因注释
    _gene_df = pd.DataFrame({
        "gene_id": adata.var_names.astype(str),
        "gene_short_name": adata.var_names.astype(str),
    })
    _gene_path = os.path.join(MONOCLE3_WORK_DIR, "gene_anno.csv")
    _gene_df.to_csv(_gene_path, index=False)
    print(f"基因注释已导出: {_gene_path}  ({len(_gene_df)} 基因)")

    # UMAP 坐标（让 Monocle3 复用 Python 侧的 embedding）
    if "X_umap" in adata.obsm:
        _umap = pd.DataFrame(
            adata.obsm["X_umap"][:, :2],
            index=adata.obs_names.astype(str),
            columns=["UMAP1", "UMAP2"],
        )
        _umap.index.name = "cell_id"
        _umap.reset_index().to_csv(
            os.path.join(MONOCLE3_WORK_DIR, "existing_umap.csv"), index=False
        )
        print(f"UMAP 坐标已导出")
    else:
        print("WARNING: X_umap 不存在，Monocle3 将自行计算 UMAP")
else:
    if not RUN_MONOCLE3:
        print("RUN_MONOCLE3=False，跳过数据导出")
    elif not _monocle3_available:
        print("Monocle3 不可用，跳过数据导出")


In [ ]:
# Monocle3 R 脚本——内联生成然后 subprocess 调用。
# 逻辑完整复用 10_pseudotime.ipynb 的已验证 R 桥逻辑：
# 去掉 graph_test 和 find_gene_modules（纯轨迹推断，不做差异基因），
# 复用 Python 侧的 UMAP embedding。

if RUN_MONOCLE3 and _monocle3_available:
    _r_script = os.path.join(MONOCLE3_WORK_DIR, "run_monocle3.R")
    _r_code = '''#!/usr/bin/env Rscript
suppressPackageStartupMessages({
  library(monocle3)
  library(Matrix)
  library(data.table)
  library(igraph)
})

args <- commandArgs(trailingOnly = TRUE)
input_mtx    <- args[1]
input_meta   <- args[2]
input_gene   <- args[3]
input_umap   <- args[4]
output_prefix<- args[5]
num_dim      <- as.integer(args[6])
ncores       <- as.integer(args[7])

# 读入数据
expr <- readMM(input_mtx)
cell_meta <- fread(input_meta, data.table = FALSE)
rownames(cell_meta) <- cell_meta$cell_id
gene_anno <- fread(input_gene, data.table = FALSE)
rownames(gene_anno) <- gene_anno$gene_id

cds <- new_cell_data_set(
  expression_data = expr,
  cell_metadata = cell_meta,
  gene_metadata = gene_anno
)

# PCA 预降维
n_dim <- min(as.integer(num_dim), nrow(cds) - 1L, ncol(cds) - 1L)
n_dim <- max(2L, n_dim)
cds <- estimate_size_factors(cds)
cds <- preprocess_cds(cds, method = "PCA", num_dim = n_dim)

# 注入已有 UMAP，让 Monocle3 直接复用 Python 侧 embedding
if (file.exists(input_umap)) {
  umap_df <- fread(input_umap, data.table = FALSE)
  rownames(umap_df) <- umap_df$cell_id
  umap_mat <- as.matrix(umap_df[colnames(cds), c("UMAP1", "UMAP2")])
  rownames(umap_mat) <- colnames(cds)
  reducedDims(cds)$UMAP <- umap_mat
} else {
  cds <- reduce_dimension(cds, reduction_method = "UMAP",
                          preprocess_method = "PCA",
                          umap.fast_sgd = TRUE, cores = ncores)
}

# 聚类 + 图学习
cds <- cluster_cells(cds, reduction_method = "UMAP")
cds <- learn_graph(cds, use_partition = TRUE, close_loop = FALSE)

# 从 root cluster 选择 root cells —— 用 adata.obs 中标记的 root_cluster
cell_meta <- colData(cds)
if ("root_cluster" %in% colnames(cell_meta)) {
  top_cluster <- args[8]
  root_cells <- rownames(cell_meta)[as.character(cell_meta[["root_cluster"]]) == top_cluster]
} else {
  # fallback: 选 pseudotime 方向最接近起点的细胞
  g <- principal_graph(cds)[["UMAP"]]
  root_cells <- NULL
}

if (length(root_cells) > 0) {
  cds <- order_cells(cds, root_cells = root_cells)
} else {
  cds <- order_cells(cds)
}

# 提取叶节点与分支点
g <- principal_graph(cds)[["UMAP"]]
all_vertices <- igraph::V(g)$name
leaf_nodes  <- all_vertices[igraph::degree(g) == 1]
branch_nodes <- all_vertices[igraph::degree(g) > 2]

# 细胞-to-vertex 映射
closest_raw <- principal_graph_aux(cds)[["UMAP"]]$pr_graph_cell_proj_closest_vertex
if (is.matrix(closest_raw) || is.data.frame(closest_raw)) {
  if (!is.null(rownames(closest_raw)) && all(colnames(cds) %in% rownames(closest_raw))) {
    closest_vertex <- as.character(closest_raw[colnames(cds), 1])
  } else if (nrow(closest_raw) == ncol(cds)) {
    closest_vertex <- as.character(closest_raw[, 1])
  } else {
    closest_vertex <- as.character(closest_raw[1, ])
  }
} else {
  closest_vertex <- as.character(closest_raw)
}
names(closest_vertex) <- colnames(cds)

# 数字 vertex 名 -> Y_<index> 映射
if (!all(closest_vertex[!is.na(closest_vertex)] %in% all_vertices)) {
  if (all(grepl("^[0-9]+$", closest_vertex[!is.na(closest_vertex)]))) {
    closest_vertex <- paste0("Y_", closest_vertex)
  }
}

# 输出结果
pseudotime_vec <- pseudotime(cds)
cluster_vec <- tryCatch(as.character(clusters(cds)),
                        error = function(e) rep(NA_character_, ncol(cds)))
partition_vec <- tryCatch(as.character(partitions(cds)),
                          error = function(e) rep(NA_character_, ncol(cds)))

cell_out <- data.frame(
  cell_id = colnames(cds),
  pseudotime = as.numeric(pseudotime_vec),
  monocle3_cluster = cluster_vec,
  monocle3_partition = partition_vec,
  monocle3_closest_vertex = closest_vertex[colnames(cds)],
  is_leaf_direct = closest_vertex[colnames(cds)] %in% leaf_nodes,
  is_branch_direct = closest_vertex[colnames(cds)] %in% branch_nodes,
  monocle3_umap1 = reducedDims(cds)$UMAP[, 1],
  monocle3_umap2 = reducedDims(cds)$UMAP[, 2],
  stringsAsFactors = FALSE
)

write.csv(cell_out, paste0(output_prefix, "_cells.csv"), row.names = FALSE)

# 画轨迹图
png(paste0(output_prefix, "_trajectory_pseudotime.png"),
    width = 2200, height = 1800, res = 220)
print(plot_cells(cds, color_cells_by = "pseudotime",
                 label_leaves = TRUE, label_branch_points = TRUE,
                 graph_label_size = 1.5))
dev.off()

png(paste0(output_prefix, "_trajectory_partition.png"),
    width = 2200, height = 1800, res = 220)
print(plot_cells(cds, color_cells_by = "partition",
                 label_leaves = TRUE, label_branch_points = TRUE,
                 graph_label_size = 1.5))
dev.off()

cat("Monocle3 completed successfully.\\n")
'''

    # 写 R 脚本
    with open(_r_script, "w", encoding="utf-8") as _f:
        _f.write(_r_code)
    print(f"R 脚本已生成: {_r_script}")

    # 调用 Rscript
    _mtx_path = os.path.join(MONOCLE3_WORK_DIR, "counts.mtx")
    _meta_path = os.path.join(MONOCLE3_WORK_DIR, "cell_meta.csv")
    _gene_path = os.path.join(MONOCLE3_WORK_DIR, "gene_anno.csv")
    _umap_path = os.path.join(MONOCLE3_WORK_DIR, "existing_umap.csv")
    _out_prefix = os.path.join(MONOCLE3_WORK_DIR, "monocle3")

    _cmd = [
        RSCRIPT_BIN, "--vanilla", _r_script,
        _mtx_path, _meta_path, _gene_path, _umap_path, _out_prefix,
        str(MONOCLE3_NUM_DIM), str(MONOCLE3_CORES),
        str(_top_cluster),
    ]
    print(f"正在运行: {' '.join(_cmd)}")

    _env = os.environ.copy()
    _env["R_PROFILE_USER"] = ""
    _env["R_ENVIRON_USER"] = ""

    try:
        _res = subprocess.run(
            _cmd, capture_output=True, text=True,
            env=_env, timeout=1800,  # Monocle3 可能需要较长时间
        )
        # 保存 stdout/stderr
        _stdout_path = os.path.join(MONOCLE3_WORK_DIR, "stdout.log")
        _stderr_path = os.path.join(MONOCLE3_WORK_DIR, "stderr.log")
        with open(_stdout_path, "w") as _f:
            _f.write(_res.stdout or "")
        with open(_stderr_path, "w") as _f:
            _f.write(_res.stderr or "")

        if _res.returncode != 0:
            print(f"Monocle3 运行失败 (exitcode={_res.returncode})")
            print(f"  STDOUT: {_stdout_path}")
            print(f"  STDERR: {_stderr_path}")
            print(f"  STDERR 尾部: {(_res.stderr or '')[-300:]}")
            _monocle3_success = False
        else:
            print("Monocle3 运行成功")
            _monocle3_success = True
    except (OSError, FileNotFoundError) as _e:
        print(f"Monocle3 运行失败（OS/文件错误）: {_e}")
        _monocle3_success = False
    except subprocess.TimeoutExpired:
        print("Monocle3 运行超时（>30 分钟），已终止")
        _monocle3_success = False
else:
    if not RUN_MONOCLE3:
        print("RUN_MONOCLE3=False，跳过 Monocle3 运行")
    elif not _monocle3_available:
        print("Monocle3 不可用，跳过")
    _monocle3_success = False


In [ ]:
# 读取 Monocle3 结果，写回 adata.obs 和 adata.obsm。

if RUN_MONOCLE3 and _monocle3_available and _monocle3_success:
    _cells_csv = os.path.join(MONOCLE3_WORK_DIR, "monocle3_cells.csv")
    if os.path.exists(_cells_csv):
        _m3 = pd.read_csv(_cells_csv).set_index("cell_id")
        # 对齐细胞索引
        _m3 = _m3.reindex(adata.obs_names)

        if "pseudotime" in _m3.columns:
            adata.obs["pseudotime_monocle3_v1"] = pd.to_numeric(
                _m3["pseudotime"], errors="coerce"
            ).values
            _pt = adata.obs["pseudotime_monocle3_v1"]
            print(f"Monocle3 pseudotime 已写入 adata.obs")
            print(f"  均值: {_pt.mean():.3f}  |  范围: [{_pt.min():.3f}, {_pt.max():.3f}]")
            print(f"  非 NA 细胞: {_pt.notna().sum()}/{len(_pt)}")

        if "monocle3_cluster" in _m3.columns:
            adata.obs["monocle3_cluster"] = _m3["monocle3_cluster"].astype(str).values

        if "monocle3_partition" in _m3.columns:
            adata.obs["monocle3_partition"] = _m3["monocle3_partition"].astype(str).values

        if "is_leaf_direct" in _m3.columns:
            adata.obs["monocle3_is_leaf"] = _m3["is_leaf_direct"].fillna(False).values

        # Monocle3 UMAP（可能与 Python 侧 UMAP 略有差异，保存为独立 key）
        if {"monocle3_umap1", "monocle3_umap2"}.issubset(_m3.columns):
            adata.obsm["X_monocle3_umap"] = _m3[[
                "monocle3_umap1", "monocle3_umap2"
            ]].to_numpy(dtype=np.float32)
            print("Monocle3 UMAP 已写入 adata.obsm['X_monocle3_umap']")

        # 复制 R 产出的 figure 到 results/figures/
        for _fname in [
            "monocle3_trajectory_pseudotime.png",
            "monocle3_trajectory_partition.png",
        ]:
            _src = os.path.join(MONOCLE3_WORK_DIR, _fname)
            if os.path.exists(_src):
                _dst = os.path.join("results", "figures", f"10b_{_fname}")
                shutil.copy2(_src, _dst)
                print(f"Monocle3 figure 已复制: {_dst}")
    else:
        print(f"Monocle3 结果文件不存在: {_cells_csv}")
else:
    _reasons = []
    if not RUN_MONOCLE3:
        _reasons.append("RUN_MONOCLE3=False")
    if not _monocle3_available:
        _reasons.append("Monocle3 环境不可用")
    if not _monocle3_success:
        _reasons.append("Monocle3 运行失败")
    print(f"Monocle3 结果不可用（{'；'.join(_reasons)}），pseudotime_monocle3_v1 不会写入 adata.obs")


In [ ]:
# 内存自检 -- 确保 adata.X 稀疏性/精度在拟时序流程中未被破坏。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 检查新增 obs 列
_new_cols = ["pseudotime_monocle3_v1", "monocle3_cluster", "monocle3_partition",
             "monocle3_is_leaf", "root_cluster"]
for _c in _new_cols:
    if _c in adata.obs.columns:
        _v = adata.obs[_c]
        print(f"  {_c}: non-NA={_v.notna().sum()}/{len(_v)}")
    else:
        print(f"  {_c}: 未写入")


In [ ]:
# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 编号命名一致）
adata.uns["stage"] = "10b_pseudotime_monocle3"
adata.uns["version"] = "v1"
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"

# Monocle3 方法细节——嵌套 dict 记录参数（与其他 stage 的嵌套 dict 形态一致）
_monocle3_uns = {
    "method": "monocle3",
    "root_cluster": str(adata.uns.get("root_cluster", "")),
    "root_source": "upstream" if _root_from_upstream else "stem_marker_fallback",
    "epithelial_clusters": EPITHELIAL_CLUSTERS,
    "auto_subset_epithelial": AUTO_SUBSET_EPITHELIAL,
    "group_col": _group_col,
    "run_monocle3": RUN_MONOCLE3,
    "use_upstream_root": USE_UPSTREAM_ROOT,
    "monocle3_num_dim": MONOCLE3_NUM_DIM,
    "monocle3_cores": MONOCLE3_CORES,
    "monocle3_ran": _monocle3_success if "_monocle3_success" in dir() else False,
    "n_cells": adata.n_obs,
    "stem_markers_used": _present_markers,
}
adata.uns["10b_pseudotime_monocle3_v1"] = _monocle3_uns
print(f"追踪字段已写入: stage={adata.uns['stage']}  version={adata.uns['version']}  "
      f"status={adata.uns['status']}")

In [ ]:
# 写出 checkpoint。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")


In [ ]:
# 释放内存。
del adata
gc.collect()
print("内存已释放")
